In [ ]:
%pip install transformers datasets torch fastapi uvicorn pandas

In [ ]:
%pip install transformers datasets torch pandas

In [10]:
from google.colab import files
uploaded = files.upload()

Saving nagpur_infrastructure_balanced_1500_rows.csv to nagpur_infrastructure_balanced_1500_rows.csv


In [11]:
import pandas as pd

df = pd.read_csv("nagpur_infrastructure_balanced_1500_rows.csv")
print(df.columns)

Index(['Department', 'Complaint'], dtype='object')


In [12]:
import pandas as pd
from datasets import Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)

# Load dataset
df = pd.read_csv("nagpur_infrastructure_balanced_1500_rows.csv")

# 🔥 Rename columns to standard format (IMPORTANT)
df = df.rename(columns={
    "Complaint": "complaint",
    "Department": "department"
})

# Convert to HF dataset
dataset = Dataset.from_pandas(df)

# Load tokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(
        batch["complaint"],
        padding=True,
        truncation=True,
        max_length=128
    )

dataset = dataset.map(tokenize, batched=True)

# Encode labels
labels = dataset.unique("department")
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

def encode(example):
    example["labels"] = label2id[example["department"]]
    return example

dataset = dataset.map(encode)

# Load model
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./dept_model",
    per_device_train_batch_size=16,
    num_train_epochs=2,
    logging_dir="./logs",
    save_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

trainer.train()

model.save_pretrained("./dept_model")
tokenizer.save_pretrained("./dept_model")

print("Training complete!")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

...

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete!


In [13]:
import os
print(os.listdir())

['.config', 'dept_model', 'nagpur_infrastructure_balanced_1500_rows.csv', 'sample_data']


In [17]:
from google.colab import drive
drive.mount('/content/drive')

model.save_pretrained("/content/drive/MyDrive/dept_model")
tokenizer.save_pretrained("/content/drive/MyDrive/dept_model")

Mounted at /content/drive


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/dept_model/tokenizer_config.json',
 '/content/drive/MyDrive/dept_model/tokenizer.json')

In [18]:
dept_tokenizer = DistilBertTokenizerFast.from_pretrained("/content/drive/MyDrive/dept_model")
dept_model = DistilBertForSequenceClassification.from_pretrained("/content/drive/MyDrive/dept_model")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [14]:
import torch
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    pipeline
)

# Load department model
dept_tokenizer = DistilBertTokenizerFast.from_pretrained("dept_model")
dept_model = DistilBertForSequenceClassification.from_pretrained("dept_model")
dept_model.eval()

# Load emotion model
emotion_classifier = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base"
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

model.safetensors:   0%|          | 0.00/329M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [15]:
EMOTION_MAP = {
    "anger": "Angry",
    "fear": "Concerned",
    "sadness": "Frustrated",
    "joy": "Appreciative",
    "neutral": "Neutral",
    "disgust": "Furious"
}

def predict_department(text):
    inputs = dept_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = dept_model(**inputs)
        predicted_id = torch.argmax(outputs.logits, dim=1).item()

    return dept_model.config.id2label[predicted_id]


def predict_sentiment(text):
    # Placeholder since emotion classifier could not be loaded.
    return "Neutral"


def predict_api(complaint_text):
    return {
        "department": predict_department(complaint_text),
        "sentiment": predict_sentiment(complaint_text)
    }

In [19]:
predict_api("The street lights are not working and I am very angry")

{'department': 'Electrical Department', 'sentiment': 'Neutral'}